In [10]:
print("hello world")

hello world


**[0. 경로설정]**
- 본인 이니셜 경로 지정(data, processed, input)
- OUTPUT_PATH로 페이지 범위 설정 "merge_streamer_data_*p_*p.csv"

In [11]:
import pandas as pd
from pathlib import Path

# ============================================================
# [0. 경로 설정]
# - 현재 notebook을 각자 이니셜 폴더(notebooks/LSB 등) 안에서 실행하는 기준
# ============================================================

BASE_DIR = Path.cwd()   # 현재 실행 위치: notebooks/LSB
PROCESSED_DIR = BASE_DIR / "data" / "processed"
INPUT_DIR = BASE_DIR / "data" / "input"

OUTPUT_PATH = BASE_DIR / "merge_streamer_data_121p_122p.csv"

print("BASE_DIR      :", BASE_DIR)
print("PROCESSED_DIR :", PROCESSED_DIR)
print("INPUT_DIR     :", INPUT_DIR)
print("OUTPUT_PATH   :", OUTPUT_PATH)

BASE_DIR      : c:\Users\ozo10\sparta17_data11_final_project\notebooks\OHC
PROCESSED_DIR : c:\Users\ozo10\sparta17_data11_final_project\notebooks\OHC\data\processed
INPUT_DIR     : c:\Users\ozo10\sparta17_data11_final_project\notebooks\OHC\data\input
OUTPUT_PATH   : c:\Users\ozo10\sparta17_data11_final_project\notebooks\OHC\merge_streamer_data_121p_122p.csv


In [12]:
import re

# ============================================================
# [1. 병합 대상 processed CSV 선택]
# - TARGET_START ~ TARGET_END 범위에 해당하는 processed CSV만 선택
# - 파일명 예시: final_streamer_data_p61_p90.csv
# ============================================================

def extract_page_range(filename: str):
    m = re.search(r"_p(\d+)_p(\d+)", filename)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None

# 병합할 페이지 범위 설정
TARGET_START = 121
TARGET_END = 122

processed_files = []

for f in PROCESSED_DIR.glob("final_streamer_data_p*_p*.csv"):
    page_range = extract_page_range(f.name)
    if page_range is None:
        continue

    start, end = page_range

    # 지정한 범위 안에 완전히 포함되는 파일만 선택
    if start >= TARGET_START and end <= TARGET_END:
        processed_files.append(f)

processed_files = sorted(processed_files)

print("=" * 60)
print("병합 대상 파일 수:", len(processed_files))
for f in processed_files:
    print("-", f.name)

병합 대상 파일 수: 1
- final_streamer_data_p121_p122.csv


In [13]:
# ============================================================
# [2. processed CSV 병합]
# - 선택된 processed CSV들을 하나로 병합
# - 완전히 동일한 행은 중복 제거
# - 최종 CSV 저장 후 파일 크기(MB) 출력
# ============================================================

dfs = []

for file in processed_files:
    df = pd.read_csv(file, encoding="utf-8-sig")
    dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)

# 전체 행 수
total_rows = len(merged_df)

# 완전히 동일한 행 제거 기준
unique_rows = len(merged_df.drop_duplicates())

# 중복 행 개수
duplicate_count = total_rows - unique_rows

print("=" * 60)
print("전체 행 수:", total_rows)
print("중복 제거 후 행 수:", unique_rows)
print("중복된 행 개수:", duplicate_count)

# 중복 제거 적용
merged_df = merged_df.drop_duplicates()

# CSV 저장
merged_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# 파일 크기 (MB)
file_size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)

print("=" * 60)
print("병합 완료")
print("shape:", merged_df.shape)
print("저장 위치:", OUTPUT_PATH)
print(f"파일 크기: {file_size_mb:.2f} MB")

전체 행 수: 479659
중복 제거 후 행 수: 479659
중복된 행 개수: 0
병합 완료
shape: (479659, 11)
저장 위치: c:\Users\ozo10\sparta17_data11_final_project\notebooks\OHC\merge_streamer_data_121p_122p.csv
파일 크기: 40.21 MB


In [14]:
import re

# ============================================================
# [3. 랭킹 CSV 선택]
# - 병합한 페이지 범위와 동일한 input 랭킹 CSV만 선택
# - 예: TARGET_START=61, TARGET_END=90 이면
#   해당 범위 안에 포함되는 LSB_streamers_p*_p*.csv만 선택
# ============================================================

def extract_page_range(filename: str):
    m = re.search(r"_p(\d+)_p(\d+)", filename)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None

ranking_files = []

for f in INPUT_DIR.glob(f"{MEMBER_TAG}_streamers_p*_p*.csv"):
    page_range = extract_page_range(f.name)
    if page_range is None:
        continue

    start, end = page_range

    # 지정한 범위 안에 완전히 포함되는 파일만 선택
    if start >= TARGET_START and end <= TARGET_END:
        ranking_files.append(f)

ranking_files = sorted(ranking_files)

print("=" * 60)
print("비교 대상 랭킹 CSV 수:", len(ranking_files))
for f in ranking_files:
    print("-", f.name)

NameError: name 'MEMBER_TAG' is not defined

In [ ]:
# ============================================================
# [4. 랭킹 CSV와 병합 CSV 비교]
# - 지정한 페이지 범위의 랭킹 CSV 기준으로
#   병합본에 없는 streamer_name 확인
# ============================================================

rank_dfs = []

for file in ranking_files:
    df = pd.read_csv(file, encoding="utf-8-sig")
    rank_dfs.append(df)

rank_df = pd.concat(rank_dfs, ignore_index=True)

# 공백 제거 후 비교용 컬럼 생성
rank_df["streamer_name_clean"] = rank_df["streamer_name"].astype(str).str.strip()
merged_df["streamer_name_clean"] = merged_df["streamer_name"].astype(str).str.strip()

rank_names = set(rank_df["streamer_name_clean"].dropna())
merged_names = set(merged_df["streamer_name_clean"].dropna())

missing_names = sorted(rank_names - merged_names)
extra_names = sorted(merged_names - rank_names)

print("=" * 60)
print("랭킹 CSV 기준 streamer_name 수:", len(rank_names))
print("병합 CSV 기준 streamer_name 수:", len(merged_names))
print("병합본에 없는 streamer_name 수:", len(missing_names))
print("랭킹엔 없지만 병합본엔 있는 streamer_name 수:", len(extra_names))

랭킹 CSV 기준 streamer_name 수: 1198
병합 CSV 기준 streamer_name 수: 1198
병합본에 없는 streamer_name 수: 0
랭킹엔 없지만 병합본엔 있는 streamer_name 수: 0


In [ ]:
# ============================================================
# [5. 누락된 streamer_name 상세 확인]
# - 병합본에 없는 streamer_name이
#   몇 페이지 몇 위였는지 확인
# ============================================================

missing_detail = rank_df[
    rank_df["streamer_name_clean"].isin(missing_names)
][["page", "rank", "streamer_name", "platform_key", "channel_id"]].drop_duplicates()

missing_detail = missing_detail.sort_values(
    by=["page", "rank"],
    key=lambda col: pd.to_numeric(col, errors="coerce")
).reset_index(drop=True)

print("=" * 60)
print("누락 상세 목록")
display(missing_detail)

# 앞부분만 보고 싶으면
# display(missing_detail.head(50))

누락 상세 목록


,page,rank,streamer_name,platform_key,channel_id


In [ ]:
# ============================================================
# [6. 누락 목록 CSV 저장]
# ============================================================

MISSING_OUTPUT_PATH = BASE_DIR / f"missing_streamer_names_{TARGET_START}p_{TARGET_END}p.csv"

missing_detail.to_csv(MISSING_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("누락 목록 저장 위치:", MISSING_OUTPUT_PATH)

In [1]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv('fancafe_final.csv')

# 2. 각 컬럼별 결측치 개수 확인하기
missing_values = df.isnull().sum()
print(missing_values)

streamer_name     0
cafe_id           0
fancafe_url       0
member_count     23
post_count       23
dtype: int64


In [2]:
# 결측치가 하나라도 포함된 데이터(행)만 골라서 보기
missing_rows = df[df.isnull().any(axis=1)]
missing_rows

,streamer_name,cafe_id,fancafe_url,member_count,post_count
271,징버거☆,ArticleList.nhn,https://cafe.naver.com/ArticleList.nhn?search....,NaN,NaN
2153,해냄,haenaemguild,https://cafe.naver.com/haenaemguild,NaN,NaN
2160,RuriHana,rurihana,https://cafe.naver.com/rurihana,NaN,NaN
2161,가날이,ganal,https://cafe.naver.com/ganal,NaN,NaN
2259,Red rose 로즈양,rose04,https://cafe.naver.com/rose04,NaN,NaN
2349,단여,ArticleList.nhn,https://cafe.naver.com/ArticleList.nhn?search....,NaN,NaN
2360,메디 MediBael,medibael,https://cafe.naver.com/medibael,NaN,NaN
2364,박유넬,yunell,https://cafe.naver.com/yunell,NaN,NaN
2365,반유,gomjellim,https://cafe.naver.com/gomjellim,NaN,NaN
2385,아델 포드,adelfan,https://cafe.naver.com/adelfan,NaN,NaN


In [3]:
# 1. 이름이 정확히 일치하는 데이터 찾기 (예: '다롬' 찾기)
#exact_match = df[df['streamer_name'] == '']
#display(exact_match)

# 2. 이름의 일부만 포함되어 있어도 찾기 (예: '마'가 들어간 사람 모두 찾기) -> 요게 더 유용할 때가 많아요!
partial_match = df[df['streamer_name'].str.contains('릴', na=False)]
display(partial_match)

,streamer_name,cafe_id,fancafe_url,member_count,post_count
520,릴리나봉,lillynabong,https://cafe.naver.com/lillynabong,7.0,5.0
1622,릴리 Lily Lue,carrothouse220206,https://cafe.naver.com/carrothouse220206,1861.0,10011.0
1623,릴리작가,lilllylillly,https://cafe.naver.com/lilllylillly,74.0,121.0
1931,릴오일,llioill0219,https://cafe.naver.com/llioill0219?tc=shared_link,28.0,76.0
2188,릴리냐,lillinya,https://cafe.naver.com/lillinya,15.0,136.0
2949,해피릴리,cczhappylilly,https://cafe.naver.com/cczhappylilly,54.0,719.0
